In [1]:
from splinter import Browser
from bs4 import BeautifulSoup as soup  
import time
import json
import random
import os

In [2]:
browser = Browser('chrome')
city = "Noida"
target_cars = 1000
cars_collected = 0
total_pages = 200

In [3]:
def collect_car_links(city, total_pages, target_cars):
    cars_collected = 0
    output_file = f"car_links_{city}.txt"

    # ✅ load existing links into a set (fast O(1) lookup, no duplicates)
    existing_links = set()
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            existing_links = set(line.strip() for line in f if line.strip())
        print(f"📂 Loaded {len(existing_links)} existing links.")

    with open(output_file, "a") as f:
        for page_num in range(1, total_pages + 1):

            if page_num == 1:
                url = f"https://www.cardekho.com/used-cars+in+{city}"
            else:
                url = f"https://www.cardekho.com/used-cars+in+{city}/page-{page_num}"

            browser.visit(url)
            time.sleep(2)
            browser.execute_script("window.scrollTo(0, 1000);")
            time.sleep(1)

            current_soup = soup(browser.html, 'html.parser')

            page_links_found = 0

            for link in current_soup.find_all('a', href=True):
                href = link['href']

                if 'used-car-details' in href:
                    full_url = f"https://www.cardekho.com{href}" if href.startswith('/') else href

                    # ✅ only write if not already seen
                    if full_url not in existing_links:
                        f.write(full_url + "\n")
                        existing_links.add(full_url)  # ✅ update set immediately
                        cars_collected += 1
                        page_links_found += 1

            print(f"Page {page_num}: Saved {page_links_found} new links. Total: {cars_collected}")

            if cars_collected >= target_cars:
                print("✅ Target reached!")
                break

    print(f"💾 All links saved in {output_file}. Total unique: {len(existing_links)}")

In [4]:
collect_car_links(city, total_pages, target_cars)

Page 1: Saved 28 new links. Total: 28
Page 2: Saved 20 new links. Total: 48
Page 3: Saved 21 new links. Total: 69
Page 4: Saved 20 new links. Total: 89
Page 5: Saved 25 new links. Total: 114
Page 6: Saved 20 new links. Total: 134
Page 7: Saved 19 new links. Total: 153
Page 8: Saved 20 new links. Total: 173
Page 9: Saved 20 new links. Total: 193
Page 10: Saved 20 new links. Total: 213
Page 11: Saved 20 new links. Total: 233
Page 12: Saved 20 new links. Total: 253
Page 13: Saved 20 new links. Total: 273
Page 14: Saved 20 new links. Total: 293
Page 15: Saved 17 new links. Total: 310
Page 16: Saved 20 new links. Total: 330
Page 17: Saved 20 new links. Total: 350
Page 18: Saved 16 new links. Total: 366
Page 19: Saved 18 new links. Total: 384
Page 20: Saved 19 new links. Total: 403
Page 21: Saved 16 new links. Total: 419
Page 22: Saved 19 new links. Total: 438
Page 23: Saved 19 new links. Total: 457
Page 24: Saved 20 new links. Total: 477
Page 25: Saved 19 new links. Total: 496
Page 26: Save

In [5]:
with open(f'car_links_{city}.txt', 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

total = len(lines)
unique = len(set(lines))
duplicates = total - unique

print(f"Total lines:     {total}")
print(f"Unique lines:    {unique}")
print(f"Duplicate lines: {duplicates}")

Total lines:     1006
Unique lines:    1006
Duplicate lines: 0


## Main Extraction 

In [6]:
def scrape_car_links(city, links_filename="car_links.txt"):

    def get_browser():
        return Browser('chrome')

    # 1. Load links
    with open(links_filename, "r") as f:
        all_links = [line.strip() for line in f.readlines()]

    output_file = f"car_dataset_{city.lower()}.json"
    browser = get_browser()

    for index, link in enumerate(all_links):
        try:
            print(f"Scraping {index+1}/{len(all_links)}: {link}")

            # Visit page
            browser.visit(link)

            # Human delay + scroll
            time.sleep(random.uniform(1, 2))
            browser.execute_script("window.scrollTo(0, 600);")
            time.sleep(0.5)

            # Expand specifications
            try:
                view_all_spec_btn = browser.find_by_text('View all Specifications')
                if view_all_spec_btn:
                    browser.execute_script(
                        "arguments[0].click();",
                        view_all_spec_btn.first._element
                    )
                    print("Expanded specifications.")
                    time.sleep(0.6)
            except Exception:
                pass

            # Parse page
            page_soup = soup(browser.html, 'html.parser')

            car_data = {"url": link}

            # -------------------------
            # CAR NAME EXTRACTION
            # -------------------------
            name_tag = page_soup.find('div', class_='vehicleName')
            h1 = name_tag.find('h1') if (name_tag and name_tag.find('h1')) else page_soup.find('h1')

            if h1:
                parts = h1.get_text(separator="|", strip=True).split("|")
                car_data["car_name"] = parts[1].strip() if len(parts) >= 2 else parts[0].strip()

            # -------------------------
            # PRICE EXTRACTION
            # -------------------------
            price_div = page_soup.find('div', class_='vehiclePrice')
            if price_div:
                price_span = price_div.find('span')
                if price_span:
                    car_data["Price"] = price_span.get_text(strip=True)

            # -------------------------
            # SPECIFICATIONS EXTRACTION
            # -------------------------
            spec_items = page_soup.find_all('li', class_='gsc_col-xs-12')

            for item in spec_items:
                label_tag = item.find('div', class_='label')
                value_tag = item.find('span', class_='value-text')

                if label_tag and value_tag:
                    label = label_tag.get_text(strip=True)
                    value = value_tag.get_text(strip=True)
                    car_data[label] = value

            # Save data
            if len(car_data) > 1:
                with open(output_file, "a") as out:
                    out.write(json.dumps(car_data) + "\n")

                print(f"Saved: {car_data.get('Price','N/A')} and {len(car_data)-2} specs.")
            else:
                print(f"No data found for: {link}")

        except Exception as e:
            print(f"Serious error at {link}: {e}")

            browser.quit()
            browser = get_browser()
            time.sleep(1)
            continue

    browser.quit()

In [7]:
scrape_car_links(f"{city}", f"car_links_{city}.txt")

Scraping 1/1006: https://www.cardekho.com/used-car-details/used-Mercedes-benz-gle-300d-bsvi-cars-Gurgaon_7ad04a3e-a848-4db3-ae7d-9714f00f1c19.htm?adId=23626&adType=1
Expanded specifications.
Saved: ₹60 Lakh and 52 specs.
Scraping 2/1006: https://www.cardekho.com/used-car-details/used-Maruti-grand-vitara-alpha-plus-hybrid-cvt-bsvi-cars-Noida_4bf61afd-de77-43b5-a6ca-154668c4868a.htm?adId=23884&adType=41
Expanded specifications.
Saved: ₹13 Lakh and 41 specs.
Scraping 3/1006: https://www.cardekho.com/buy-used-car-details/used-Kia-seltos-htx-diesel-cars-Noida_02d26d2b-1887-4596-978a-17103530f37b.htm
Expanded specifications.
Saved: ₹8.31 Lakh and 44 specs.
Scraping 4/1006: https://www.cardekho.com/used-car-details/used-Volkswagen-vento-16-comfortline-cars-Noida_b10e2c3d-4629-4bcb-ae82-11e8a5044bf8.htm
Expanded specifications.
Saved: ₹4.95 Lakh and 53 specs.
Scraping 5/1006: https://www.cardekho.com/buy-used-car-details/used-Honda-city-15-v-at-cars-Noida_891c688a-8be7-4767-8684-f50b2e502ad9.h